In [ ]:
import cobra
from cobra.io import load_model, read_sbml_model
from cobra.sampling import sample, OptGPSampler, ACHRSampler

In [ ]:
glc_exc = 'EX_glc(e)'
gln_exc = 'EX_gln_L(e)'
lac_exc = 'EX_lac_D(e)'
nh4_exc = 'EX_nh4(e)'
biomass_rxn = 'biomass_cho'
product_rxn = 'DM_igg[g]'

In [ ]:
# function to constrain the gsmm with the exchange fluxes
def constrain_gsmm(model, glc_uptake, gln_uptake, lac_exchange, nh4_exchange, biomass_prod = np.nan, product_prod= np.nan):
    # Set bounds for glucose exchange
    model.reactions.get_by_id(glc_exc).lower_bound = glc_uptake  # uptake is negative
    model.reactions.get_by_id(glc_exc).upper_bound = glc_uptake

    # Set bounds for glutamine exchange
    model.reactions.get_by_id(gln_exc).lower_bound = gln_uptake  # uptake is negative
    model.reactions.get_by_id(gln_exc).upper_bound = gln_uptake

    # Set bounds for lactate exchange
    model.reactions.get_by_id(lac_exc).lower_bound = lac_exchange
    model.reactions.get_by_id(lac_exc).upper_bound = lac_exchange  # secretion is positive

    # Set bounds for ammonium exchange
    model.reactions.get_by_id(nh4_exc).lower_bound = nh4_exchange
    model.reactions.get_by_id(nh4_exc).upper_bound = nh4_exchange  # secretion is positive

    # Set bounds for biomass production if the value is not NaN
    if not np.isnan(biomass_prod):
        model.reactions.get_by_id(biomass_rxn).lower_bound = biomass_prod
        model.reactions.get_by_id(biomass_rxn).upper_bound = biomass_prod

    # Set bounds for product formation if value is not NaN
    if not np.isnan(product_prod):
        model.reactions.get_by_id(product_rxn).lower_bound = product_prod
        model.reactions.get_by_id(product_rxn).upper_bound = product_prod

    return model

In [ ]:
import pickle
import numpy as np
from cobra.sampling import OptGPSampler

# Molecular weight constants (module-level, computed once)
DA = 1.066e-24  # 1 Da = 1.066e-24 g
IGG_MOLECULAR_WEIGHT = 50 * DA * 1000  # g --- 50 kDa per molecule
IGG_MOLAR_WEIGHT = IGG_MOLECULAR_WEIGHT * 6.022e23  # g/mol
IGG_MW_MG = IGG_MOLAR_WEIGHT * 1000  # mg/mol


def constrain_and_sample_batch(
    model,
    sp_data,
    batch_label,
    n_samples=5000,
    thinning=500,
    processes=10,
    output_dir=".",
    verbose=True
):
    """
    Constrain a GSMM using time-resolved specific rates for a given batch,
    then sample the resulting flux polytope at each timepoint.

    Parameters
    ----------
    model : cobra.Model
        Base (unconstrained) genome-scale metabolic model. A fresh copy is
        made for each timepoint.
    sp_data : dict
        Dictionary with keys:
            'times'    : array-like, shape (n_steps,)
            'X_states' : array-like, shape (n_steps, 7)
                         columns = [cdw, glc, gln, lac, amm, Ab, vol]
            'rates'    : array-like, shape (n_steps, n_targets)
                         columns = [r_cdw, r_glc, r_gln, r_lac, r_nh4, r_Ab]
    batch_label : str
        Identifier used in print statements and output filenames
        (e.g. "top_batch2").
    n_samples : int, default 5000
        Number of samples to draw with OptGPSampler.
    thinning : int, default 500
        Thinning parameter for OptGPSampler.
    processes : int, default 10
        Number of parallel processes for sampling.
    output_dir : str, default "."
        Directory where sampled flux pickle files are saved.
    verbose : bool, default True
        Whether to print progress/diagnostic information.

    Returns
    -------
    constrained_models : list of cobra.Model
        The constrained model at each timepoint (in order of `times`).
    sampled_fluxes_list : list of pandas.DataFrame
        Sampled flux DataFrames for timepoints with an optimal solution.
        Timepoints with non-optimal solutions are skipped (not appended).
    solution_statuses : list of str
        Optimization status at each timepoint (same order as `times`).
    """

    times = sp_data['times']
    X_states = sp_data['X_states']
    sp_rates = sp_data['rates']  # shape: (n_steps, n_targets)

    constrained_models = []
    sampled_fluxes_list = []
    solution_statuses = []

    for t_idx in range(len(times)):
        x_state = X_states[t_idx]
        cdw, glc, gln, lac, amm, Ab, vol = x_state
        r_cdw, r_glc, r_gln, r_lac, r_nh4, r_Ab = sp_rates[t_idx]

        # Convert specific rates to fluxes (mmol/gDW/h)
        glc_uptake = r_glc
        gln_uptake = r_gln
        lac_exchange = r_lac
        nh4_exchange = r_nh4
        biomass_prod = r_cdw
        product_prod = r_Ab / IGG_MW_MG  # convert to mmol/gDW/h

        if verbose:
            print(f'Constraints at time {times[t_idx]} h:')
            print(f'  Glc uptake: {glc_uptake} mmol/gDW/h')
            print(f'  Gln uptake: {gln_uptake} mmol/gDW/h')
            print(f'  Lac exchange: {lac_exchange} mmol/gDW/h')
            print(f'  NH4 exchange: {nh4_exchange} mmol/gDW/h')
            print(f'  Biomass production: {biomass_prod} /h')
            print(f'  Product production: {product_prod} mmol/gDW/h')

        # Constrain model
        constrained_model = constrain_gsmm(
            model.copy(), glc_uptake, gln_uptake, lac_exchange, nh4_exchange
        )
        solution = constrained_model.optimize()
        solution_statuses.append(solution.status)

        if verbose:
            print(f"Time {times[t_idx]} h: Optimization status: {solution.status}, "
                  f"Objective value: {solution.objective_value}")
            print('-' * 60)

        constrained_models.append(constrained_model)

        if solution.status == 'optimal':
            sampler = OptGPSampler(constrained_model, thinning=thinning, processes=processes)
            samples = sampler.sample(n_samples)
            sampled_fluxes_list.append(samples)

            # Save samples to pickle
            out_path = f'{output_dir}/{batch_label}_sampled_fluxes_timepoint_{t_idx}.pkl'
            with open(out_path, 'wb') as f:
                pickle.dump(samples, f)

            if verbose:
                print(f"Saved sampled fluxes for timepoint {t_idx} to '{out_path}'")
        else:
            if verbose:
                print(f"Skipping sampling for time {times[t_idx]} h due to non-optimal solution.")

    return constrained_models, sampled_fluxes_list, solution_statuses


# ===== Example usage, replicating the original script's behavior =====
if __name__ == "__main__":
    # For the 2nd best batch (top_ids[1]):
    high_yield_batch_id = top_ids[1]
    sp_data = top_sp_rates[high_yield_batch_id]

    constrained_models, sampled_fluxes_list, statuses = constrain_and_sample_batch(
        model=model,
        sp_data=sp_data,
        batch_label="top_batch2",
        n_samples=5000,
        thinning=500,
        processes=10,
        output_dir=".",
    )

    # To repeat for additional top batches, just loop:
    # for rank, batch_id in enumerate(top_ids[:N]):
    #     sp_data = top_sp_rates[batch_id]
    #     constrain_and_sample_batch(
    #         model=model,
    #         sp_data=sp_data,
    #         batch_label=f"top_batch{rank+1}",
    #     )

In [ ]:
for rank, batch_id in enumerate(top_ids[:5]):  # top 5 batches
    sp_data = top_sp_rates[batch_id]
    constrained_models, sampled_fluxes_list, statuses = constrain_and_sample_batch(
        model=model,
        sp_data=sp_data,
        batch_label=f"top_batch{rank+1}",
        output_dir="./sampled_fluxes",
    )